# AE4 · Métricas y visualización de datos

**Proyecto Integrador III – Analítica de Datos (IU Digital) · Grupo 1**

Este notebook es el **taller de los gráficos** del dashboard. Aquí se construye y se revisa cada métrica y cada gráfico
con la clase `DashboardAE4` (`src/AE4/dashboard_ae4.py`). 

## 1. Rutas y reuso de los datos de AE3

In [1]:
import sys
from pathlib import Path

import pandas as pd
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_columns", 50)

# Raíz del repositorio: la primera carpeta hacia arriba que contenga src/AE3 y src/AE4
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "AE3").exists() and (p / "src" / "AE4").exists())
DIR_AE3 = RAIZ / "src" / "AE3" / "datos"
DIR_AE4 = RAIZ / "src" / "AE4"
DIR_DATOS_APP = DIR_AE4 / "datos"
DIR_RESULTADOS = DIR_AE4 / "resultados"

for ruta in (RAIZ / "src" / "clases", DIR_AE4):
    if str(ruta) not in sys.path:
        sys.path.insert(0, str(ruta))

# Reuso de las clases de AE1–AE3 (se importan los módulos directamente)
from reportes import Reportes
from preprocesador import PreprocesadorHouseSales
from enriquecedor_nemotron import EnriquecedorNemotron
from dashboard_ae4 import (DashboardAE4, PREGUNTAS, PREGUNTA_NEGOCIO, ORDEN_TAMANOS, ORDEN_CALIFICACION,
                           ORDEN_VISTA, ARCHIVO_DATOS, ARCHIVO_AUDITORIA, ARCHIVO_AUDITORIA_COLUMNAS,
                           ARCHIVO_DICCIONARIO)

print("Raíz del repositorio encontrada:", RAIZ.name)

Raíz del repositorio encontrada: repo_prueba


In [2]:
crudo = pd.read_csv(DIR_AE3 / "reg_cat_house_sales_raw.csv")
enriquecido = pd.read_csv(DIR_AE3 / "reg_cat_house_sales_enriquecido.csv", parse_dates=["Fecha de venta"])

print(f"Crudo (AE2):        {crudo.shape[0]:,} filas × {crudo.shape[1]} columnas")
print(f"Enriquecido (AE3):  {enriquecido.shape[0]:,} filas × {enriquecido.shape[1]} columnas")
assert enriquecido.shape == (21521, 43), "El dataset de AE3 no tiene la forma esperada"
assert enriquecido.isna().sum().sum() == 0, "Hay valores nulos en el dataset de AE3"

Crudo (AE2):        21,613 filas × 18 columnas
Enriquecido (AE3):  21,521 filas × 43 columnas


## 2. Copia de datos para la app, auditoría y diccionario

El workflow de GitHub Actions borra y regenera `src/AE3/datos` en cada `push`. Por eso la app lee **su propia copia**
en `src/AE4/datos/`. Aquí también se reutilizan dos clases de AE3:

- `PreprocesadorHouseSales.comparar_antes_despues`: la auditoría antes vs. después que muestra la pestaña **Proyecto**.
- `EnriquecedorNemotron.obtener_diccionario_variables_nuevas`: el diccionario de las variables creadas (no llama a la API).

In [3]:
DashboardAE4.exportar_datos(DIR_AE3 / "reg_cat_house_sales_enriquecido.csv", DIR_DATOS_APP / ARCHIVO_DATOS)

auditoria_global, auditoria_columnas = PreprocesadorHouseSales().comparar_antes_despues(crudo, enriquecido)
auditoria_global.to_csv(DIR_DATOS_APP / ARCHIVO_AUDITORIA, index=False)
auditoria_columnas.to_csv(DIR_DATOS_APP / ARCHIVO_AUDITORIA_COLUMNAS, index=False)

diccionario = EnriquecedorNemotron().obtener_diccionario_variables_nuevas()
diccionario.to_csv(DIR_DATOS_APP / ARCHIVO_DICCIONARIO, index=False)

# Reporte descriptivo del dataset que usa la app (clase Reportes de AE1–AE3)
Reportes().guardar_reporte(enriquecido, DIR_RESULTADOS / "resumen_datos_dashboard_ae4.txt",
                           dataset_name="inria-soda/tabular-benchmark", config="reg_cat_house_sales", split="train",
                           muestra="Dataset enriquecido de AE3 usado por el dashboard de AE4")

for archivo in sorted(DIR_DATOS_APP.glob("*.csv")):
    print(f"{archivo.name:45s} {archivo.stat().st_size / 1024:10,.1f} KB")
auditoria_global

auditoria_columnas.csv                               3.8 KB
auditoria_global.csv                                 1.0 KB
diccionario_variables.csv                            3.3 KB
reg_cat_house_sales_enriquecido.csv              8,620.6 KB


,Indicador,Antes (Crudo),Después (Preparado/Enriquecido),Diferencia,Explicación
0,Total de filas (registros),21613,21521,-92,Depuración justificada de 92 viviendas con hab...
1,Total de columnas (variables),18,43,25,Conservación de las 18 variables base (renombr...
2,Valores nulos totales,0,0,0,Se garantiza 0 % de valores faltantes tanto en...
3,Filas exactamente duplicadas,0,0,0,Sin duplicados antes ni después de las transfo...
4,Columnas numéricas,18,27,9,Vista al agua pasa a texto ('Sí'/'No'); se aña...
5,Columnas categóricas / texto,0,15,15,"Se agregan categorías de tamaño, calificación,..."
6,Columnas de fecha (datetime),0,1,1,Construcción de 'Fecha de venta' a partir de a...


## 3. Pregunta de negocio → indicador → visual → decisión

Cada pregunta lleva a una decisión. El gráfico se escoge por la tarea que debe cumplir: comparar grupos (barras),
ver una tendencia en el tiempo (líneas), ver una parte del total (torta), cruzar dos dimensiones (mapa de calor)
o ubicar en el territorio (mapa).

In [4]:
print("Pregunta de negocio:", PREGUNTA_NEGOCIO)
pd.DataFrame(PREGUNTAS).T.rename(columns={"titulo": "Pregunta", "indicador": "Indicador",
                                          "visual": "Visual", "decision": "Decisión"})

Pregunta de negocio: ¿Cómo varía el precio de venta de las viviendas vendidas en el Condado de King entre mayo de 2014 y mayo de 2015 según su tamaño, su calificación de construcción, su ubicación frente al agua y su antigüedad o remodelación?


,Pregunta,Indicador,Visual,Decisión
P1,Variación del precio según tamaño habitable y ...,"Mediana del precio (USD) por tamaño, habitacio...",Barras de mediana por grupo; barras de correla...,Usar rangos de referencia por tamaño para la p...
P2,Diferencias de precio según nivel de calificac...,Mediana del precio (USD) y número de viviendas...,Barras de mediana y de conteo; línea por grado...,Usar la calificación como criterio principal p...
P3,Prima descriptiva de precio por vista al mar (...,Número de viviendas y mediana del precio (USD ...,"Barras de conteo, de mediana y de USD/m²; mapa...",Tratar la vista al agua como un segmento apart...
P4,"Variación del precio por subregión, barrios, c...",Mediana del precio por m² (USD/m²) por subregi...,"Barras ordenadas, mapa de calor, mapa de preci...",Priorizar la ubicación en la matriz de referen...
P5,Variación del precio según década de construcc...,Participación (%) de remodeladas y mediana del...,Torta de participación; barras agrupadas por d...,Considerar la remodelación como valor adiciona...
P6,Dinámica mensual del volumen de ventas y los p...,"Ventas (n), mediana mensual (USD) y su variaci...",Líneas de ventas y de mediana; barras de varia...,Planear cuándo hay más movimiento de viviendas...


## 4. KPI de contexto

In [5]:
dash = DashboardAE4.desde_carpeta(DIR_DATOS_APP)
d = dash.datos
pd.DataFrame([(k, v[0]) for k, v in dash.kpis(d).items()], columns=["Indicador", "Valor"])

,Indicador,Valor
0,Viviendas,21.521
1,Precio típico (mediana),USD 450.000
2,Mediana USD/m²,USD 2.631
3,Bien calificadas (8+),48 %
4,Con vista al agua,"0,73 %"
5,Remodeladas,"4,2 %"


## 5. Verificación: las tablas del dashboard coinciden con las de AE3

El dashboard calcula sus tablas con `Reportes.tabla_estadistica_grupo`, la misma función que generó las tablas de AE3.
Si las medianas no coinciden, la celda se detiene.

In [6]:
comparaciones = {
    "tabla_p1_tamano_vivienda.csv": "Categoría de tamaño",
    "tabla_p2_calificacion_constructiva.csv": "Nivel de calificación",
    "tabla_p3_vista_mar_lago.csv": "Vista al mar o lago",
    "tabla_p4_subregiones_barrios.csv": "Municipio / Subregión",
    "tabla_p5b_resumen_remodelacion.csv": "Condición de remodelación",
    "tabla_p6_periodo_ventas.csv": "Periodo de venta",
}
filas = []
for archivo, columna in comparaciones.items():
    ae3 = pd.read_csv(DIR_AE3 / archivo).set_index(columna)
    ae4 = dash.tabla_estadistica(d, columna).set_index(columna)
    iguales = (ae3["Mediana (USD)"].sort_index() == ae4["Mediana (USD)"].sort_index()).all() and \
              (ae3["Viviendas (n)"].sort_index() == ae4["Viviendas (n)"].sort_index()).all()
    filas.append({"Tabla AE3": archivo, "Agrupación": columna, "¿Coincide?": "Sí" if iguales else "No"})
verificacion = pd.DataFrame(filas)
assert (verificacion["¿Coincide?"] == "Sí").all(), "Hay diferencias con las tablas de AE3"
verificacion

,Tabla AE3,Agrupación,¿Coincide?
0,tabla_p1_tamano_vivienda.csv,Categoría de tamaño,Sí
1,tabla_p2_calificacion_constructiva.csv,Nivel de calificación,Sí
2,tabla_p3_vista_mar_lago.csv,Vista al mar o lago,Sí
3,tabla_p4_subregiones_barrios.csv,Municipio / Subregión,Sí
4,tabla_p5b_resumen_remodelacion.csv,Condición de remodelación,Sí
5,tabla_p6_periodo_ventas.csv,Periodo de venta,Sí


## 6. Gráficos por pregunta

Para los mapas se usa una muestra de 2.000 viviendas, solo para que el notebook no pese demasiado.
La app dibuja todas las viviendas.

In [7]:
muestra_mapa = d.sample(2000, random_state=1)

### P1 · Variación del precio según tamaño habitable y capacidad (habitaciones y baños)

- **Indicador:** mediana del precio (USD) por tamaño, habitaciones y baños; correlación r con el precio.
- **Por qué este visual:** barras para comparar grupos; barras de correlación para ver qué característica pesa más.

In [8]:
dash.tabla_estadistica(d, "Categoría de tamaño", ORDEN_TAMANOS)

,Categoría de tamaño,Viviendas (n),Participación (%),Media (USD),Mediana (USD),Moda (USD),Desv. Estándar (USD),Coef. Variación (%),Q1 (USD),Q3 (USD),RIC (USD),Mínimo (USD),Máximo (USD),Mediana (USD/m²),RIC (USD/m²)
0,1. Pequeña (<130 m²),4984,23.16,337555.0,320000.0,350000.0,132875.0,39.36,235000.0,415500.0,180500.0,78000.0,1050000.0,3118.48,1827.47
1,2. Mediana (130–199 m²),8073,37.51,436258.0,412000.0,450000.0,169295.0,38.81,302200.0,535000.0,232800.0,128000.0,1500000.0,2529.60,1409.32
2,3. Grande (200–279 m²),5435,25.25,603087.0,565997.0,650000.0,248948.0,41.28,420000.0,725000.0,305000.0,194250.0,2400000.0,2431.31,1237.86
3,4. Muy grande (≥280 m²),3029,14.07,1043963.0,869950.0,850000.0,620720.0,59.46,680000.0,1229000.0,549000.0,274975.0,7700000.0,2621.96,1323.07


In [9]:
dash.figura_p1_tamano(d).show()
dash.figura_p1_habitaciones(d).show()
dash.figura_p1_banos(d).show()
dash.figura_p1_correlacion(d).show()

### P2 · Diferencias de precio según nivel de calificación constructiva

- **Indicador:** mediana del precio y número de viviendas por nivel y por grado (4–13).
- **Por qué este visual:** barras para comparar niveles, línea para ver el salto grado a grado, mapa para ubicar las bien calificadas.

In [10]:
dash.tabla_estadistica(d, "Nivel de calificación", ORDEN_CALIFICACION)

,Nivel de calificación,Viviendas (n),Participación (%),Media (USD),Mediana (USD),Moda (USD),Desv. Estándar (USD),Coef. Variación (%),Q1 (USD),Q3 (USD),RIC (USD),Mínimo (USD),Máximo (USD),Mediana (USD/m²),RIC (USD/m²)
0,1. Baja (4–6),2251,10.46,296139.0,270000.0,250000.0,123371.0,41.66,210000.0,360000.0,150000.0,78000.0,1200000.0,2568.81,1630.56
1,2. Estándar (7),8956,41.62,402577.0,375000.0,350000.0,155761.0,38.69,285000.0,485000.0,200000.0,90000.0,2050000.0,2455.21,1469.00
2,3. Media-Alta (8–9),8679,40.33,612442.0,562000.0,550000.0,272649.0,44.52,427500.0,730000.0,302500.0,140000.0,3070000.0,2690.94,1360.35
3,4. Alta / Lujo (10–13),1635,7.60,1257962.0,1038000.0,1150000.0,707064.0,56.21,812750.0,1488000.0,675250.0,316000.0,7700000.0,3047.59,1604.23


In [11]:
dash.figura_p2_mediana(d).show()
dash.figura_p2_conteo(d).show()
dash.figura_p2_grado(d).show()
dash.figura_p2_mapa(muestra_mapa).show()

### P3 · Prima de precio por vista al mar (Puget Sound) vs. lago interior

- **Indicador:** número de viviendas y mediana del precio (USD y USD/m²) por tipo de vista.
- **Por qué este visual:** el conteo muestra que el grupo es muy pequeño; las barras de mediana muestran la prima; el mapa, dónde están.

In [12]:
dash.tabla_estadistica(d, "Vista al mar o lago", ORDEN_VISTA)

,Vista al mar o lago,Viviendas (n),Participación (%),Media (USD),Mediana (USD),Moda (USD),Desv. Estándar (USD),Coef. Variación (%),Q1 (USD),Q3 (USD),RIC (USD),Mínimo (USD),Máximo (USD),Mediana (USD/m²),RIC (USD/m²)
0,Sin vista directa al agua,21364,99.27,532481.0,450000.0,450000.0,341765.0,64.18,321382.0,640000.0,318618.0,78000.0,7700000.0,2623.86,1449.36
1,Vista a lago / río interior,81,0.38,2191251.0,1955000.0,2000000.0,1005717.0,45.90,1517000.0,2750000.0,1233000.0,570000.0,5300000.0,6575.26,1984.31
2,Vista al mar (Puget Sound),76,0.35,1194955.0,908250.0,750000.0,993262.0,83.12,639875.0,1387425.0,747550.0,285000.0,7062500.0,4301.60,2440.22


In [13]:
dash.figura_p3_conteo(d).show()
dash.figura_p3_mediana(d).show()
dash.figura_p3_m2(d).show()
dash.figura_p3_mapa(d).show()

### P4 · Variación del precio por subregión, barrios, cercanía comercial y entorno

- **Indicador:** mediana del precio por m² (USD/m²) por subregión, tamaño y anillo urbano; mediana por barrio.
- **Por qué este visual:** barras ordenadas para el ranking, mapa de calor para cruzar subregión y tamaño, mapa para ver el territorio.

In [14]:
dash.tabla_estadistica(d, "Municipio / Subregión").sort_values("Mediana (USD/m²)", ascending=False)

,Municipio / Subregión,Viviendas (n),Participación (%),Media (USD),Mediana (USD),Moda (USD),Desv. Estándar (USD),Coef. Variación (%),Q1 (USD),Q3 (USD),RIC (USD),Mínimo (USD),Máximo (USD),Mediana (USD/m²),RIC (USD/m²)
3,Seattle,6327,29.40,580423.0,505000.0,450000.0,343874.0,59.25,384000.0,675000.0,291000.0,81000.0,7700000.0,3470.95,1535.02
0,Eastside (Bellevue/Kirkland/Redmond),5805,26.97,753034.0,640000.0,550000.0,469488.0,62.35,490000.0,850000.0,360000.0,130000.0,7062500.0,2960.06,977.99
5,Vashon / Islas,87,0.40,493128.0,475000.0,295000.0,217771.0,44.16,329750.0,580000.0,250250.0,160000.0,1379900.0,2664.86,1079.39
2,North King County,2146,9.97,484068.0,434500.0,350000.0,246140.0,50.85,357000.0,535875.0,178875.0,90000.0,4500000.0,2527.13,740.28
1,Foothills / Este Semirrural,1291,6.00,469108.0,431000.0,425000.0,220581.0,47.02,320000.0,561500.0,241500.0,119500.0,2140000.0,2222.34,673.22
4,South King County,5865,27.25,326207.0,295000.0,250000.0,146258.0,44.84,244000.0,368000.0,124000.0,78000.0,2510000.0,1774.88,561.95


In [15]:
dash.figura_p4_subregion(d).show()
dash.figura_p4_calor(d).show()
dash.figura_p4_anillo(d).show()
dash.figura_p4_entorno(d).show()
dash.figura_p4_mapa_m2(muestra_mapa).show()
dash.tabla_p4_barrios(d, 10)

,Barrio / Sector,Municipio / Subregión,viviendas,mediana,media,mediana_m2
20,Medina / Clyde Hill,Eastside (Bellevue/Kirkland/Redmond),403,1020000.0,1.268250e+06,4479.170
8,Downtown Bellevue,Eastside (Bellevue/Kirkland/Redmond),243,987000.0,1.246056e+06,4657.220
36,University District / Laurelhurst,Seattle,471,805000.0,9.796525e+05,4265.000
21,Mercer Island,Eastside (Bellevue/Kirkland/Redmond),548,683500.0,8.461956e+05,3275.570
30,Sammamish,Eastside (Bellevue/Kirkland/Redmond),886,680000.0,7.220005e+05,2718.250
11,Eastgate / Factoria,Eastside (Bellevue/Kirkland/Redmond),658,647500.0,7.591515e+05,3107.540
25,Queen Anne / Magnolia,Seattle,1343,630000.0,7.215733e+05,4071.950
28,Redmond,Eastside (Bellevue/Kirkland/Redmond),1062,607250.0,6.550892e+05,2979.180
18,Kirkland,Eastside (Bellevue/Kirkland/Redmond),516,575000.0,7.267461e+05,3128.935
1,Ballard / Crown Hill - Seattle,Seattle,348,555500.0,6.172400e+05,3702.365


### P5 · Variación del precio según década de construcción y condición de remodelación

- **Indicador:** participación (%) de remodeladas y mediana del precio por década y condición.
- **Por qué este visual:** torta para la parte del total; barras agrupadas para comparar remodelada vs. no remodelada en la misma década.

In [16]:
dash.tabla_estadistica(d, ["Década de construcción", "Condición de remodelación"])

,Década de construcción,Condición de remodelación,Viviendas (n),Participación (%),Media (USD),Mediana (USD),Moda (USD),Desv. Estándar (USD),Coef. Variación (%),Q1 (USD),Q3 (USD),RIC (USD),Mínimo (USD),Máximo (USD),Mediana (USD/m²),RIC (USD/m²)
0,1900s,No remodelada,540,2.51,610874.0,545000.0,600000.0,380192.0,62.24,402375.0,700250.0,297875.0,90000.0,3300000.0,3827.09,1800.36
1,1900s,Remodelada,102,0.47,760917.0,657500.0,550000.0,435903.0,57.29,458875.0,868750.0,409875.0,200000.0,2350000.0,3623.28,1614.33
2,1910s,No remodelada,682,3.17,559119.0,502500.0,400000.0,335983.0,60.09,345250.0,680000.0,334750.0,83000.0,3200000.0,3587.88,1774.61
3,1910s,Remodelada,116,0.54,775563.0,607500.0,450000.0,799589.0,103.10,423000.0,925125.0,502125.0,152900.0,7700000.0,3631.67,2103.64
4,1920s,No remodelada,1066,4.95,591007.0,531750.0,425000.0,350011.0,59.22,397250.0,686125.0,288875.0,110000.0,3000000.0,3683.01,1595.84
5,1920s,Remodelada,117,0.54,662237.0,630000.0,400000.0,314708.0,47.52,411500.0,860000.0,448500.0,134000.0,1740000.0,3261.73,1508.13
6,1930s,No remodelada,437,2.03,576852.0,460000.0,235000.0,435510.0,75.50,293000.0,700000.0,407000.0,85000.0,3650000.0,3412.13,1789.00
7,1930s,Remodelada,85,0.39,757519.0,571500.0,550000.0,605771.0,79.97,375000.0,895000.0,520000.0,152275.0,3400000.0,3060.36,1984.28
8,1940s,No remodelada,1580,7.34,425458.0,380000.0,350000.0,236750.0,55.65,268000.0,525000.0,257000.0,78000.0,2544750.0,3100.55,1730.09
9,1940s,Remodelada,155,0.72,692735.0,496000.0,400000.0,715632.0,103.31,331000.0,805500.0,474500.0,135000.0,7062500.0,3088.05,1821.75


In [17]:
dash.figura_p5_torta(d).show()
dash.figura_p5_decadas(d).show()

### P6 · Dinámica mensual del volumen de ventas y los precios

- **Indicador:** ventas (n), mediana mensual (USD) y su variación frente al mes anterior.
- **Por qué este visual:** líneas para la evolución en el tiempo; barras para la variación; barras apiladas para la composición.

In [18]:
dash.tabla_estadistica(d, "Periodo de venta")

,Periodo de venta,Viviendas (n),Participación (%),Media (USD),Mediana (USD),Moda (USD),Desv. Estándar (USD),Coef. Variación (%),Q1 (USD),Q3 (USD),RIC (USD),Mínimo (USD),Máximo (USD),Mediana (USD/m²),RIC (USD/m²)
0,2014-05,1763,8.19,548799.0,465000.0,300000.0,356665.0,64.99,326050.0,651000.0,324950.0,78000.0,3710000.0,2631.20,1395.19
1,2014-06,2168,10.07,558310.0,465000.0,440000.0,388468.0,69.58,330000.0,665000.0,335000.0,92000.0,7062500.0,2673.14,1415.39
2,2014-07,2205,10.25,545785.0,468000.0,625000.0,346660.0,63.52,325000.0,655000.0,330000.0,86500.0,3800000.0,2601.80,1395.76
3,2014-08,1932,8.98,537355.0,442536.0,375000.0,370886.0,69.02,325000.0,640000.0,315000.0,107000.0,5570000.0,2643.68,1381.09
4,2014-09,1767,8.21,530096.0,450000.0,425000.0,372921.0,70.35,325000.0,625000.0,300000.0,85000.0,6885000.0,2616.37,1404.01
5,2014-10,1865,8.67,540807.0,449000.0,400000.0,400012.0,73.97,315000.0,640000.0,325000.0,89000.0,7700000.0,2622.99,1536.74
6,2014-11,1407,6.54,522734.0,435000.0,290000.0,355903.0,68.08,309975.0,616250.0,306275.0,82000.0,3850000.0,2562.79,1397.05
7,2014-12,1464,6.80,525219.0,433250.0,500000.0,353872.0,67.38,312375.0,630000.0,317625.0,95000.0,3300000.0,2510.55,1393.08
8,2015-01,973,4.52,526933.0,440000.0,330000.0,366401.0,69.53,311000.0,625000.0,314000.0,99000.0,3567000.0,2520.99,1467.35
9,2015-02,1242,5.77,509685.0,428000.0,325000.0,326156.0,63.99,310000.0,600000.0,290000.0,89950.0,3065000.0,2547.71,1494.07


In [19]:
dash.figura_p6_ventas(d).show()
dash.figura_p6_mediana(d).show()
dash.figura_p6_variacion(d).show()
dash.figura_p6_apilado(d).show()

## 7. Respuestas de negocio calculadas

No hay cifras escritas a mano: si cambian los datos o los filtros, las respuestas cambian solas.

In [20]:
pd.set_option("display.max_colwidth", None)
pd.DataFrame(dash.respuestas(d))[["id", "respuesta", "decision"]]

,id,respuesta,decision
0,P1,"La mediana pasa de USD 320.000 en la categoría más pequeña a USD 869.950 en la más grande (2,7 veces). El área habitable es la característica más asociada al precio (r = 0.70, fuerte).",Usar rangos de referencia por tamaño para la primera estimación del avalúo.
1,P2,"10.314 viviendas (48 %) tienen calificación 8 o más. Su mediana es USD 609.675, frente a USD 350.000 del resto. Calificación y área van juntas (r = 0.76).",Usar la calificación como criterio principal para ubicar una vivienda en un rango.
2,P3,"Solo 157 de 21.521 viviendas (0,73 %) tienen vista al agua. Mediana sin vista directa al agua: USD 450.000 (n = 21364); vista a lago / río interior: USD 1.955.000 (n = 81); vista al mar (puget sound): USD 908.250 (n = 76).","Tratar la vista al agua como un segmento aparte, no como regla general del mercado."
3,P4,El m² más caro está en Seattle (USD 3.471/m²) y el más barato en South King County (USD 1.775/m²).,Priorizar la ubicación en la matriz de referencia geográfica; barrios como orientación.
4,P5,"Solo el 4,2 % está remodelado. Mediana remodeladas USD 600.500 frente a USD 449.500 no remodeladas. En 9 de 9 décadas comparables (con al menos 10 casas en cada grupo) la remodelada tiene mayor precio típico.",Considerar la remodelación como valor adicional al comparar viviendas de la misma época.
5,P6,Más ventas en 2015-04 (2.224); menos en 2015-05 (642). La mediana mensual va de USD 428.000 a USD 478.250. Mayo de 2015 es parcial (hasta el 27).,Planear cuándo hay más movimiento de viviendas y revisar precios con ritmo trimestral.


In [21]:
# Ejemplo de filtro (lo mismo que hace la barra lateral de la app): solo Seattle
seattle = dash.filtrar(ORDEN_TAMANOS, ORDEN_CALIFICACION, ["Seattle"],
                       (d["Fecha de venta"].min(), d["Fecha de venta"].max()))
print(len(seattle), "viviendas en Seattle")
print(dash.respuesta("P1", seattle))

6327 viviendas en Seattle
La mediana pasa de USD 395.000 en la categoría más pequeña a USD 1.165.000 en la más grande (2,9 veces). El área habitable es la característica más asociada al precio (r = 0.75, fuerte).


## 8. Prueba automática de la app antes de publicar

Se ejecuta la app completa sin navegador, sin filtros y con filtros. Si algo falla, aparece aquí antes de publicar.

In [22]:
from streamlit.testing.v1 import AppTest

at = AppTest.from_file(str(DIR_AE4 / "app.py"), default_timeout=180).run()
print("Errores:", [e.value for e in at.exception])
print("KPI:", [(m.label, m.value) for m in at.metric])
print("Pestañas:", len(at.tabs))
assert not at.exception

at.sidebar.multiselect[2].set_value(["Seattle"]).run()
print("Con filtro Seattle:", [(m.label, m.value) for m in at.metric][:2])
assert not at.exception
print("OK: la app pasa las pruebas")

2026-11-13 10:25:30.737 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Errores: []
KPI: [('Viviendas', '21.521'), ('Precio típico (mediana)', 'USD 450.000'), ('Mediana USD/m²', 'USD 2.631'), ('Bien calificadas (8+)', '48 %'), ('Con vista al agua', '0,73 %'), ('Remodeladas', '4,2 %')]
Pestañas: 9


Con filtro Seattle: [('Viviendas', '6.327'), ('Precio típico (mediana)', 'USD 505.000')]
OK: la app pasa las pruebas


## 9. Cómo abrir la app

Desde la raíz del repositorio:

```bash
streamlit run src/AE4/app.py
```

En Streamlit Community Cloud: repositorio del grupo, rama `main`, *Main file path* `src/AE4/app.py`, Python 3.12.

In [23]:
ABRIR_EN_NAVEGADOR = False   # ← cámbialo a True para abrir la app desde aquí

if ABRIR_EN_NAVEGADOR:
    import subprocess
    subprocess.Popen([sys.executable, "-m", "streamlit", "run", str(DIR_AE4 / "app.py")], cwd=RAIZ)